# Reddit Resume Comment Thread Scraper (No Authentication Required)

This notebook scrapes resumes from Reddit comment threads (like resume advice threads) using Reddit's JSON API (no credentials needed!).

It extracts resumes from:
- Imgur links in comments (single images and albums)
- Google Drive links in comments
- Direct image links

**Use case**: Resume advice threads where people post their resumes in comments for feedback.

**Important**: This preserves existing files in the folder - only new resumes will be added!

In [1]:
import requests
import re
import os
import json
import time
from pathlib import Path
from datetime import datetime
from urllib.parse import urlparse, urljoin
import pandas as pd
from tqdm import tqdm

print("✓ All packages imported successfully!")
print("Note: No Reddit API credentials needed for this scraper!")

✓ All packages imported successfully!
Note: No Reddit API credentials needed for this scraper!


In [2]:
# Setup requests session with proper headers
# Reddit's JSON API is public and doesn't require authentication!

session = requests.Session()
session.headers.update({
    'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
})

print("✓ HTTP session configured!")
print("✓ Ready to scrape Reddit (no credentials needed)")

✓ HTTP session configured!
✓ Ready to scrape Reddit (no credentials needed)


In [3]:
# Create directories for storing resumes
output_dir = Path("resumes")
output_dir.mkdir(exist_ok=True)

# Directory structure:
# resumes/
#   ├── files/          # All downloaded resume files
#   └── metadata.json   # Metadata about each resume

files_dir = output_dir / "files"
files_dir.mkdir(exist_ok=True)

print(f"Output directory: {output_dir.absolute()}")
print(f"Files directory: {files_dir.absolute()}")

Output directory: /Users/anirudhannabathula/Desktop/ds3/25-26_Projects/TalentLens_Public/data/reddit/resumes
Files directory: /Users/anirudhannabathula/Desktop/ds3/25-26_Projects/TalentLens_Public/data/reddit/resumes/files


In [4]:
def extract_imgur_urls(text):
    """Extract imgur URLs from text (single images and albums)"""
    if not text:
        return []
    
    imgur_patterns = [
        r'https?://(?:i\.)?imgur\.com/([a-zA-Z0-9]+)\.?(?:png|jpg|jpeg|gif|pdf)?',
        r'https?://imgur\.com/(?:a/|gallery/)([a-zA-Z0-9]+)',
    ]
    
    urls = []
    for pattern in imgur_patterns:
        matches = re.findall(pattern, text)
        urls.extend(matches)
    
    return list(set(urls))  # Remove duplicates

def extract_google_drive_urls(text):
    """Extract Google Drive file IDs from text"""
    if not text:
        return []
    
    # Patterns for Google Drive links
    drive_patterns = [
        r'drive\.google\.com/file/d/([a-zA-Z0-9_-]+)',
        r'drive\.google\.com/open\?id=([a-zA-Z0-9_-]+)',
        r'docs\.google\.com/document/d/([a-zA-Z0-9_-]+)',
    ]
    
    file_ids = []
    for pattern in drive_patterns:
        matches = re.findall(pattern, text)
        file_ids.extend(matches)
    
    return list(set(file_ids))  # Remove duplicates

def get_imgur_direct_link(imgur_id):
    """Convert imgur ID to direct download link"""
    # Try common image extensions
    extensions = ['jpg', 'png', 'jpeg', 'pdf']
    for ext in extensions:
        url = f"https://i.imgur.com/{imgur_id}.{ext}"
        try:
            response = session.head(url, timeout=5)
            if response.status_code == 200:
                return url
        except:
            continue
    return None

def get_google_drive_direct_link(file_id):
    """Convert Google Drive file ID to direct download link"""
    # Google Drive direct download link format
    return f"https://drive.google.com/uc?export=download&id={file_id}"

def download_file(url, output_path, timeout=30):
    """Download a file from URL to output_path"""
    try:
        response = session.get(url, timeout=timeout, stream=True, allow_redirects=True)
        response.raise_for_status()
        
        # Check if it's actually an image/pdf
        content_type = response.headers.get('content-type', '')
        if not any(t in content_type.lower() for t in ['image', 'pdf', 'octet-stream']):
            print(f"Skipping non-image/pdf content: {content_type}")
            return False
        
        with open(output_path, 'wb') as f:
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)
        
        # Check if file size is reasonable (> 1KB, < 50MB)
        file_size = output_path.stat().st_size
        if file_size < 1024 or file_size > 50 * 1024 * 1024:
            output_path.unlink()  # Delete the file
            return False
        
        return True
    except Exception as e:
        print(f"Error downloading {url}: {str(e)}")
        return False

def get_file_extension_from_url(url):
    """Extract file extension from URL"""
    path = urlparse(url).path
    ext = os.path.splitext(path)[1]
    if ext:
        return ext.lower()
    return '.jpg'  # Default to .jpg

def fetch_reddit_json(url, params=None):
    """Fetch JSON data from Reddit's API"""
    try:
        # Add .json to the URL if not present
        if not url.endswith('.json'):
            url = url.rstrip('/') + '.json'
        
        response = session.get(url, params=params, timeout=10)
        response.raise_for_status()
        time.sleep(2)  # Be respectful to Reddit's servers
        return response.json()
    except Exception as e:
        print(f"Error fetching {url}: {str(e)}")
        return None

def get_existing_file_ids():
    """Get set of already downloaded file identifiers to avoid duplicates"""
    existing_ids = set()
    if files_dir.exists():
        for file in files_dir.iterdir():
            if file.is_file():
                # Extract identifier from filename (e.g., "imgur_abc123" or "gdrive_xyz789")
                name = file.stem
                existing_ids.add(name)
    return existing_ids

print("✓ Helper functions loaded successfully!")

✓ Helper functions loaded successfully!


In [5]:
def process_comment(comment_data, existing_ids):
    """Process a Reddit comment and extract resume links"""
    comment_info = {
        'comment_id': comment_data.get('id', ''),
        'author': comment_data.get('author', '[deleted]'),
        'created_utc': comment_data.get('created_utc', 0),
        'created_date': datetime.fromtimestamp(comment_data.get('created_utc', 0)).strftime('%Y-%m-%d %H:%M:%S'),
        'body': comment_data.get('body', ''),
        'score': comment_data.get('score', 0),
        'permalink': f"https://reddit.com{comment_data.get('permalink', '')}",
        'downloaded_files': []
    }
    
    downloaded_count = 0
    comment_id = comment_info['comment_id']
    body = comment_info['body']
    
    # Extract imgur links
    imgur_ids = extract_imgur_urls(body)
    for imgur_id in imgur_ids:
        file_id = f"imgur_{imgur_id}"
        if file_id in existing_ids:
            continue  # Skip if already downloaded
            
        direct_url = get_imgur_direct_link(imgur_id)
        if direct_url:
            ext = get_file_extension_from_url(direct_url)
            filename = f"{file_id}{ext}"
            output_path = files_dir / filename
            
            if download_file(direct_url, output_path):
                comment_info['downloaded_files'].append({
                    'filename': filename,
                    'source': 'imgur',
                    'url': direct_url,
                    'imgur_id': imgur_id
                })
                existing_ids.add(file_id)
                downloaded_count += 1
    
    # Extract Google Drive links
    gdrive_ids = extract_google_drive_urls(body)
    for gdrive_id in gdrive_ids:
        file_id = f"gdrive_{gdrive_id}"
        if file_id in existing_ids:
            continue  # Skip if already downloaded
            
        direct_url = get_google_drive_direct_link(gdrive_id)
        filename = f"{file_id}.pdf"  # Default to PDF for Google Drive
        output_path = files_dir / filename
        
        if download_file(direct_url, output_path):
            comment_info['downloaded_files'].append({
                'filename': filename,
                'source': 'google_drive',
                'url': direct_url,
                'drive_id': gdrive_id
            })
            existing_ids.add(file_id)
            downloaded_count += 1
    
    # Extract direct image links
    direct_image_urls = re.findall(r'(https?://[^\s]+\.(?:jpg|jpeg|png|pdf|gif))', body, re.IGNORECASE)
    for url in direct_image_urls:
        # Skip if it's already an imgur or drive link
        if 'imgur.com' in url or 'drive.google.com' in url:
            continue
            
        # Create a unique ID from URL
        url_hash = str(hash(url))[-8:]
        file_id = f"direct_{url_hash}"
        if file_id in existing_ids:
            continue
            
        ext = get_file_extension_from_url(url)
        filename = f"{file_id}{ext}"
        output_path = files_dir / filename
        
        if download_file(url, output_path):
            comment_info['downloaded_files'].append({
                'filename': filename,
                'source': 'direct_link',
                'url': url
            })
            existing_ids.add(file_id)
            downloaded_count += 1
    
    return comment_info, downloaded_count

def flatten_comments(comments_json, all_comments=None):
    """Recursively flatten nested comment structure"""
    if all_comments is None:
        all_comments = []
    
    for comment in comments_json:
        if comment['kind'] == 't1':  # t1 is a comment
            all_comments.append(comment['data'])
            
            # Process replies if they exist
            if 'replies' in comment['data'] and comment['data']['replies']:
                if isinstance(comment['data']['replies'], dict):
                    reply_data = comment['data']['replies'].get('data', {})
                    children = reply_data.get('children', [])
                    flatten_comments(children, all_comments)
    
    return all_comments

print("✓ Comment processor loaded!")

✓ Comment processor loaded!


In [10]:
# Configuration for scraping
# List of Reddit thread URLs to scrape
THREAD_URLS = [
    # Add more thread URLs here as needed
    #"https://www.reddit.com/r/cscareerquestions/comments/1qrwgup/resume_advice_thread_january_31_2026/",
    "https://www.reddit.com/r/cscareerquestions/comments/1qo7rb7/resume_advice_thread_january_27_2026/",
]

print(f"Configuration:")
print(f"Number of threads to scrape: {len(THREAD_URLS)}")
print(f"\nThreads:")
for i, url in enumerate(THREAD_URLS, 1):
    print(f"  {i}. {url}")
print(f"\nNote: This will preserve existing files and only download new resumes")

Configuration:
Number of threads to scrape: 1

Threads:
  1. https://www.reddit.com/r/cscareerquestions/comments/1qo7rb7/resume_advice_thread_january_27_2026/

Note: This will preserve existing files and only download new resumes


In [11]:
# Main scraping execution
print(f"Starting to scrape {len(THREAD_URLS)} thread(s)...\n")

# Get existing file IDs to avoid duplicates
existing_ids = get_existing_file_ids()
print(f"Found {len(existing_ids)} existing files - these will be skipped\n")

all_comments_data = []
total_files_downloaded = 0
comments_with_files = 0

for thread_num, thread_url in enumerate(THREAD_URLS, 1):
    print(f"{'='*70}")
    print(f"Thread {thread_num}/{len(THREAD_URLS)}: {thread_url}")
    print(f"{'='*70}")
    
    # Fetch the thread data
    thread_data = fetch_reddit_json(thread_url)
    
    if not thread_data or len(thread_data) < 2:
        print(f"✗ Failed to fetch thread data")
        continue
    
    # Reddit returns array: [0] = post, [1] = comments
    post_data = thread_data[0]['data']['children'][0]['data']
    comments_data = thread_data[1]['data']['children']
    
    print(f"Post: {post_data.get('title', 'No title')}")
    print(f"Author: {post_data.get('author', '[deleted]')}")
    print(f"Score: {post_data.get('score', 0)}")
    print(f"\nFlattening comment tree...")
    
    # Flatten nested comments
    all_comments = flatten_comments(comments_data)
    print(f"Found {len(all_comments)} total comments")
    
    # Process each comment
    print(f"\nProcessing comments for resume links...\n")
    for comment_data in tqdm(all_comments, desc=f"Thread {thread_num}"):
        try:
            comment_info, file_count = process_comment(comment_data, existing_ids)
            
            if file_count > 0:
                all_comments_data.append(comment_info)
                comments_with_files += 1
                total_files_downloaded += file_count
                
                author = comment_info['author']
                preview = comment_info['body'][:80].replace('\n', ' ')
                print(f"✓ u/{author}: Downloaded {file_count} file(s)")
                print(f"   Preview: {preview}...")
                
        except Exception as e:
            print(f"✗ Error processing comment: {str(e)}")
            continue
    
    print(f"\nCompleted thread {thread_num}/{len(THREAD_URLS)}")
    print(f"Files downloaded from this thread: {total_files_downloaded}\n")

print(f"\n{'='*70}")
print(f"SCRAPING COMPLETE!")
print(f"{'='*70}")
print(f"Total comments processed: {sum(len(flatten_comments(fetch_reddit_json(url)[1]['data']['children'])) for url in THREAD_URLS if fetch_reddit_json(url))}")
print(f"Comments with resumes: {comments_with_files}")
print(f"New files downloaded: {total_files_downloaded}")
print(f"Total files in folder: {len(existing_ids) + total_files_downloaded}")
print(f"Files saved to: {files_dir.absolute()}")
print(f"{'='*70}")

Starting to scrape 1 thread(s)...

Found 15 existing files - these will be skipped

Thread 1/1: https://www.reddit.com/r/cscareerquestions/comments/1qo7rb7/resume_advice_thread_january_27_2026/
Post: Resume Advice Thread - January 27, 2026
Author: CSCQMods
Score: 2

Flattening comment tree...
Found 16 total comments

Processing comments for resume links...



Thread 1:  12%|█▎        | 2/16 [00:01<00:08,  1.63it/s]

✓ u/redundant_engineer: Downloaded 1 file(s)
   Preview: Unexpectedly got hit with the big redundancy hammer 5 months into my first post-...


Thread 1: 100%|██████████| 16/16 [00:02<00:00,  7.71it/s]



Completed thread 1/1
Files downloaded from this thread: 1


SCRAPING COMPLETE!
Total comments processed: 16
Comments with resumes: 1
New files downloaded: 1
Total files in folder: 17
Files saved to: /Users/anirudhannabathula/Desktop/ds3/25-26_Projects/TalentLens_Public/data/reddit/resumes/files


In [12]:
# Save metadata to JSON file
metadata_file = output_dir / "comments_metadata.json"

# Load existing metadata if it exists
existing_metadata = []
if metadata_file.exists():
    with open(metadata_file, 'r', encoding='utf-8') as f:
        existing_metadata = json.load(f)

# Combine existing and new metadata
all_metadata = existing_metadata + all_comments_data

with open(metadata_file, 'w', encoding='utf-8') as f:
    json.dump(all_metadata, f, indent=2, ensure_ascii=False)

print(f"Metadata saved to: {metadata_file.absolute()}")
print(f"Total comments with resumes: {len(all_metadata)}")

# Also create a CSV for easier analysis
if all_comments_data:
    comments_df = pd.DataFrame([
        {
            'comment_id': comment['comment_id'],
            'author': comment['author'],
            'created_date': comment['created_date'],
            'score': comment['score'],
            'num_files': len(comment['downloaded_files']),
            'permalink': comment['permalink'],
            'body_preview': comment['body'][:100].replace('\n', ' ')
        }
        for comment in all_comments_data
    ])
    
    csv_file = output_dir / "comments_summary.csv"
    
    # Append to existing CSV if it exists
    if csv_file.exists():
        existing_df = pd.read_csv(csv_file)
        comments_df = pd.concat([existing_df, comments_df], ignore_index=True)
    
    comments_df.to_csv(csv_file, index=False)
    print(f"Comments summary saved to: {csv_file.absolute()}")

Metadata saved to: /Users/anirudhannabathula/Desktop/ds3/25-26_Projects/TalentLens_Public/data/reddit/resumes/comments_metadata.json
Total comments with resumes: 1
Comments summary saved to: /Users/anirudhannabathula/Desktop/ds3/25-26_Projects/TalentLens_Public/data/reddit/resumes/comments_summary.csv


In [13]:
# Display summary statistics
print("=" * 70)
print("SUMMARY STATISTICS")
print("=" * 70)

if all_comments_data:
    print(f"\nComments with resumes (this session): {len(all_comments_data)}")
    print(f"New files downloaded (this session): {total_files_downloaded}")
    
    # File type distribution
    print(f"\n\nFile type distribution (this session):")
    file_extensions = []
    for comment in all_comments_data:
        for file_info in comment['downloaded_files']:
            ext = os.path.splitext(file_info['filename'])[1]
            file_extensions.append(ext)
    
    if file_extensions:
        ext_df = pd.DataFrame({'extension': file_extensions})
        print(ext_df['extension'].value_counts())
    
    # Source distribution
    print(f"\n\nSource distribution (this session):")
    sources = []
    for comment in all_comments_data:
        for file_info in comment['downloaded_files']:
            sources.append(file_info['source'])
    
    if sources:
        source_df = pd.DataFrame({'source': sources})
        print(source_df['source'].value_counts())
    
    # Top contributors
    print(f"\n\nTop contributors (this session):")
    authors = [comment['author'] for comment in all_comments_data]
    if authors:
        author_df = pd.DataFrame({'author': authors})
        print(author_df['author'].value_counts().head(10))
else:
    print("\nNo new resume files were found in the scraped threads.")

# Overall folder statistics
print(f"\n\n{'='*70}")
print(f"OVERALL FOLDER STATISTICS")
print(f"{'='*70}")
total_files_in_folder = len(list(files_dir.glob('*'))) if files_dir.exists() else 0
print(f"Total files in folder: {total_files_in_folder}")
print(f"Folder location: {files_dir.absolute()}")

SUMMARY STATISTICS

Comments with resumes (this session): 1
New files downloaded (this session): 1


File type distribution (this session):
extension
.jpg    1
Name: count, dtype: int64


Source distribution (this session):
source
imgur    1
Name: count, dtype: int64


Top contributors (this session):
author
redundant_engineer    1
Name: count, dtype: int64


OVERALL FOLDER STATISTICS
Total files in folder: 17
Folder location: /Users/anirudhannabathula/Desktop/ds3/25-26_Projects/TalentLens_Public/data/reddit/resumes/files


In [14]:
# Display sample comments with downloaded resumes
print("=" * 70)
print("SAMPLE COMMENTS WITH RESUMES (This Session)")
print("=" * 70)

if all_comments_data:
    for i, comment in enumerate(all_comments_data[:5]):  # Show first 5
        print(f"\n{i+1}. Comment by u/{comment['author']}")
        print(f"   Date: {comment['created_date']}")
        print(f"   Score: {comment['score']}")
        print(f"   Link: {comment['permalink']}")
        print(f"   Files downloaded ({len(comment['downloaded_files'])}):")
        for file_info in comment['downloaded_files']:
            print(f"      - {file_info['filename']} (from {file_info['source']})")
        print(f"   Comment preview: {comment['body'][:100].replace(chr(10), ' ')}...")
else:
    print("\nNo comments with downloaded files to display.")

SAMPLE COMMENTS WITH RESUMES (This Session)

1. Comment by u/redundant_engineer
   Date: 2026-01-28 10:19:26
   Score: 1
   Link: https://reddit.com/r/cscareerquestions/comments/1qo7rb7/resume_advice_thread_january_27_2026/o29cnb2/
   Files downloaded (1):
      - imgur_rTAFL5J.jpg (from imgur)
   Comment preview: Unexpectedly got hit with the big redundancy hammer 5 months into my first post-graduation job last ...


## Optional: Add More Threads

To scrape additional resume advice threads, simply add their URLs to the `THREAD_URLS` list in cell 6:

```python
THREAD_URLS = [
    "https://www.reddit.com/r/cscareerquestions/comments/1qrwgup/resume_advice_thread_january_31_2026/",
    "https://www.reddit.com/r/cscareerquestions/comments/ANOTHER_THREAD_ID/resume_thread/",
    # Add more threads here
]
```

Then re-run cells 7-10 to scrape the new threads.

### Finding Resume Threads

Good places to find resume advice threads:
- r/cscareerquestions - Weekly resume advice threads
- r/csMajors - Resume roast/review threads
- r/EngineeringResumes - Resume review threads
- r/jobs - Resume help threads

Just copy the thread URL and add it to the list!

## Setup Instructions

### 1. Install Required Packages

Simply install the required Python packages:

```bash
pip install requests pandas tqdm
```

That's it! No Reddit API credentials needed!

### 2. How it works

This scraper:
- Uses Reddit's **public JSON API** (no authentication required)
- Extracts imgur and Google Drive links from comment text
- Preserves existing files (no duplicates)
- Works with any public Reddit thread

### 3. Configure threads to scrape

Edit cell 6 and add the Reddit thread URLs you want to scrape:

```python
THREAD_URLS = [
    "https://www.reddit.com/r/cscareerquestions/comments/1qrwgup/resume_advice_thread_january_31_2026/",
    # Add more URLs here
]
```

### 4. Run the notebook

Run cells 1-10 in order to scrape and analyze the data!

In [ ]:
# Optional: Load previously saved data
# If you've already run the scraper and want to analyze the data again, run this cell

try:
    metadata_file = output_dir / "comments_metadata.json"
    with open(metadata_file, 'r') as f:
        all_comments_data = json.load(f)
    
    csv_file = output_dir / "comments_summary.csv"
    if csv_file.exists():
        comments_df = pd.read_csv(csv_file)
        print(f"Loaded {len(all_comments_data)} comments from {metadata_file.name}")
        print(f"Total files in dataset: {comments_df['num_files'].sum()}")
    else:
        print(f"Loaded {len(all_comments_data)} comments from {metadata_file.name}")
except FileNotFoundError:
    print("No saved data found. Run the scraper first.")

In [ ]:
# Create a README file in the output directory
total_files_in_folder = len(list(files_dir.glob('*'))) if files_dir.exists() else 0

readme_content = f"""# Reddit Resume Dataset - Comment Threads

## Overview
This dataset contains student resumes scraped from Reddit resume advice thread comments.

## Scraping Details
- **Source**: Reddit resume advice threads
- **Last Update**: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
- **Total Files**: {total_files_in_folder}
- **Scraped Threads**: {len(THREAD_URLS)}

## Dataset Structure

```
resumes/
├── files/                    # All downloaded resume files
├── comments_metadata.json    # Complete metadata for all comments and files
├── comments_summary.csv      # Summary table of all comments
└── README.md                # This file
```

## Metadata Fields

### comments_metadata.json
Each comment entry contains:
- `comment_id`: Reddit comment ID
- `author`: Reddit username of commenter
- `created_utc`: Unix timestamp of comment creation
- `created_date`: Human-readable date
- `body`: Full text content of the comment
- `score`: Reddit score (upvotes - downvotes)
- `permalink`: Link to the Reddit comment
- `downloaded_files`: Array of file information
  - `filename`: Name of the downloaded file
  - `source`: Where the file came from (imgur, google_drive, direct_link)
  - `url`: Original URL of the file
  - `imgur_id` or `drive_id`: Source-specific identifier

### comments_summary.csv
Simplified table with key information for quick analysis.

## File Naming Convention

Files are named using the pattern: `{{source}}_{{id}}.{{extension}}`

Examples:
- `imgur_abc123.jpg` - Image from Imgur
- `gdrive_xyz789.pdf` - File from Google Drive
- `direct_abc12345.png` - Direct image link

## Supported Link Types

- **Imgur**: Single images and albums
- **Google Drive**: Shared files and documents
- **Direct Links**: Direct image URLs (.jpg, .png, .pdf, etc.)

## Usage Notes

- All files are downloaded as-is from their sources
- Some files may be images of resumes rather than PDF documents
- File quality and format vary based on what users uploaded
- Metadata preserves attribution to original commenters
- Existing files are preserved - the scraper only adds new files

## Ethics and Privacy

- All data is publicly posted on Reddit
- Usernames are preserved as they appear publicly
- If you use this data, please respect users' privacy
- Consider reaching out to users before using their resumes for any commercial purpose

## Citation

If you use this dataset, please cite:
- Source: Reddit resume advice threads
- Collection Date: {datetime.now().strftime('%Y-%m-%d')}
- Threads scraped: {len(THREAD_URLS)}
"""

readme_path = output_dir / "README.md"
with open(readme_path, 'w', encoding='utf-8') as f:
    f.write(readme_content)

print(f"README created at: {readme_path.absolute()}")

## Optional: Auto-find Resume Advice Threads

You can automatically find recent resume advice threads from various subreddits:

```python
# Subreddits that commonly have resume threads
subreddits_to_check = ['cscareerquestions', 'csMajors', 'EngineeringResumes', 'jobs']

resume_thread_urls = []

for subreddit in subreddits_to_check:
    print(f"Searching r/{subreddit} for resume threads...")
    
    search_url = f"https://www.reddit.com/r/{subreddit}/search.json"
    params = {
        'q': 'resume advice thread OR resume review',
        'restrict_sr': 'on',
        'sort': 'new',
        't': 'month',
        'limit': 10
    }
    
    data = fetch_reddit_json(search_url, params)
    
    if data and 'data' in data:
        posts = data['data'].get('children', [])
        for post in posts:
            post_data = post['data']
            title = post_data.get('title', '')
            if 'resume' in title.lower() and ('advice' in title.lower() or 'review' in title.lower() or 'thread' in title.lower()):
                url = f"https://reddit.com{post_data.get('permalink', '')}"
                resume_thread_urls.append(url)
                print(f"  ✓ Found: {title}")

print(f"\n\nFound {len(resume_thread_urls)} resume threads:")
print("\nCopy these URLs to THREAD_URLS in cell 6:")
for url in resume_thread_urls:
    print(f'    "{url}",')
```